In [41]:
# import libraries for sentiment analysis
import pandas as pd # read dataset 
import numpy as np # numeric operations
from textblob import TextBlob # get subjectivity for each text
import re # text cleaning
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # get VADER scores for each text
from sklearn.model_selection import train_test_split # split data into train and test sets
from sklearn.metrics import classification_report # evaluate model performance
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis # classifier to predict sentiment labels

from newsapi import NewsApiClient
import pandas as pd
import yfinance as yf
from pathlib import Path

# reference : https://www.youtube.com/watch?v=4OlvGGAsj8I&t=698s


In [42]:
headlines_path = Path("training_data/djia_news.csv")
djia_path = Path("training_data/djia_price_table.csv")


df1 = pd.read_csv(headlines_path, encoding="cp1252", low_memory=False, parse_dates=["Date"]) 
df2 = pd.read_csv(djia_path, encoding="cp1252", low_memory=False, parse_dates=["Date"]) 

In [43]:
print("df1:", df1.shape)
display(df1.head(3))


df1: (1989, 27)


,Date,Label,Top1,Top2,Top3,Top4,Top5,Top6,Top7,Top8,...,Top16,Top17,Top18,Top19,Top20,Top21,Top22,Top23,Top24,Top25
0,2008-08-08,0,"b""Georgia 'downs two Russian warplanes' as cou...",b'BREAKING: Musharraf to be impeached.',b'Russia Today: Columns of troops roll into So...,b'Russian tanks are moving towards the capital...,"b""Afghan children raped with 'impunity,' U.N. ...",b'150 Russian tanks have entered South Ossetia...,"b""Breaking: Georgia invades South Ossetia, Rus...","b""The 'enemy combatent' trials are nothing but...",...,b'Georgia Invades South Ossetia - if Russia ge...,b'Al-Qaeda Faces Islamist Backlash',"b'Condoleezza Rice: ""The US would not act to p...",b'This is a busy day: The European Union has ...,"b""Georgia will withdraw 1,000 soldiers from Ir...",b'Why the Pentagon Thinks Attacking Iran is a ...,b'Caucasus in crisis: Georgia invades South Os...,b'Indian shoe manufactory - And again in a se...,b'Visitors Suffering from Mental Illnesses Ban...,"b""No Help for Mexico's Kidnapping Surge"""
1,2008-08-11,1,b'Why wont America and Nato help us? If they w...,b'Bush puts foot down on Georgian conflict',"b""Jewish Georgian minister: Thanks to Israeli ...",b'Georgian army flees in disarray as Russians ...,"b""Olympic opening ceremony fireworks 'faked'""",b'What were the Mossad with fraudulent New Zea...,b'Russia angered by Israeli military sale to G...,b'An American citizen living in S.Ossetia blam...,...,b'Israel and the US behind the Georgian aggres...,"b'""Do not believe TV, neither Russian nor Geor...",b'Riots are still going on in Montreal (Canada...,b'China to overtake US as largest manufacturer',b'War in South Ossetia [PICS]',b'Israeli Physicians Group Condemns State Tort...,b' Russia has just beaten the United States ov...,b'Perhaps *the* question about the Georgia - R...,b'Russia is so much better at war',"b""So this is what it's come to: trading sex fo..."
2,2008-08-12,0,b'Remember that adorable 9-year-old who sang a...,"b""Russia 'ends Georgia operation'""","b'""If we had no sexual harassment we would hav...","b""Al-Qa'eda is losing support in Iraq because ...",b'Ceasefire in Georgia: Putin Outmaneuvers the...,b'Why Microsoft and Intel tried to kill the XO...,b'Stratfor: The Russo-Georgian War and the Bal...,"b""I'm Trying to Get a Sense of This Whole Geor...",...,b'U.S. troops still in Georgia (did you know t...,b'Why Russias response to Georgia was right',"b'Gorbachev accuses U.S. of making a ""serious ...","b'Russia, Georgia, and NATO: Cold War Two'",b'Remember that adorable 62-year-old who led y...,b'War in Georgia: The Israeli connection',b'All signs point to the US encouraging Georgi...,b'Christopher King argues that the US and NATO...,b'America: The New Mexico?',"b""BBC NEWS | Asia-Pacific | Extinction 'by man..."


In [44]:
print("df2:", df2.shape)
display(df2.head(3))


df2: (1989, 7)


,Date,Open,High,Low,Close,Volume,Adj Close
0,2016-07-01,17924.240234,18002.380859,17916.910156,17949.369141,82160000,17949.369141
1,2016-06-30,17712.759766,17930.609375,17711.800781,17929.990234,133030000,17929.990234
2,2016-06-29,17456.019531,17704.509766,17456.019531,17694.679688,106380000,17694.679688


In [45]:
# create a new dataset merging headlines and DJIA data on Date
merge = df1.merge(df2, how="inner", on="Date")

# show the mergede dataset
merge.head(3)

,Date,Label,Top1,Top2,Top3,Top4,Top5,Top6,Top7,Top8,...,Top22,Top23,Top24,Top25,Open,High,Low,Close,Volume,Adj Close
0,2008-08-08,0,"b""Georgia 'downs two Russian warplanes' as cou...",b'BREAKING: Musharraf to be impeached.',b'Russia Today: Columns of troops roll into So...,b'Russian tanks are moving towards the capital...,"b""Afghan children raped with 'impunity,' U.N. ...",b'150 Russian tanks have entered South Ossetia...,"b""Breaking: Georgia invades South Ossetia, Rus...","b""The 'enemy combatent' trials are nothing but...",...,b'Caucasus in crisis: Georgia invades South Os...,b'Indian shoe manufactory - And again in a se...,b'Visitors Suffering from Mental Illnesses Ban...,"b""No Help for Mexico's Kidnapping Surge""",11432.089844,11759.959961,11388.040039,11734.320312,212830000,11734.320312
1,2008-08-11,1,b'Why wont America and Nato help us? If they w...,b'Bush puts foot down on Georgian conflict',"b""Jewish Georgian minister: Thanks to Israeli ...",b'Georgian army flees in disarray as Russians ...,"b""Olympic opening ceremony fireworks 'faked'""",b'What were the Mossad with fraudulent New Zea...,b'Russia angered by Israeli military sale to G...,b'An American citizen living in S.Ossetia blam...,...,b' Russia has just beaten the United States ov...,b'Perhaps *the* question about the Georgia - R...,b'Russia is so much better at war',"b""So this is what it's come to: trading sex fo...",11729.669922,11867.110352,11675.530273,11782.349609,183190000,11782.349609
2,2008-08-12,0,b'Remember that adorable 9-year-old who sang a...,"b""Russia 'ends Georgia operation'""","b'""If we had no sexual harassment we would hav...","b""Al-Qa'eda is losing support in Iraq because ...",b'Ceasefire in Georgia: Putin Outmaneuvers the...,b'Why Microsoft and Intel tried to kill the XO...,b'Stratfor: The Russo-Georgian War and the Bal...,"b""I'm Trying to Get a Sense of This Whole Geor...",...,b'All signs point to the US encouraging Georgi...,b'Christopher King argues that the US and NATO...,b'America: The New Mexico?',"b""BBC NEWS | Asia-Pacific | Extinction 'by man...",11781.700195,11782.349609,11601.519531,11642.469727,173590000,11642.469727


In [ ]:
# combine news headlines into one column 
headlines = []

# for each row in the dataframe, combine the news headlines into one string - iloc is used to access the rows and columns by index 
for row in range(0,len(merge.index)):
    headlines.append(' '.join(str(x) for x in merge.iloc[row, 2:27])) # 25 news headlines from column 2 to 26


In [47]:
# print a sample of the combined headlines 
headlines[0]

'b"Georgia \'downs two Russian warplanes\' as countries move to brink of war" b\'BREAKING: Musharraf to be impeached.\' b\'Russia Today: Columns of troops roll into South Ossetia; footage from fighting (YouTube)\' b\'Russian tanks are moving towards the capital of South Ossetia, which has reportedly been completely destroyed by Georgian artillery fire\' b"Afghan children raped with \'impunity,\' U.N. official says - this is sick, a three year old was raped and they do nothing" b\'150 Russian tanks have entered South Ossetia whilst Georgia shoots down two Russian jets.\' b"Breaking: Georgia invades South Ossetia, Russia warned it would intervene on SO\'s side" b"The \'enemy combatent\' trials are nothing but a sham: Salim Haman has been sentenced to 5 1/2 years, but will be kept longer anyway just because they feel like it." b\'Georgian troops retreat from S. Osettain capital, presumably leaving several hundred people killed. [VIDEO]\' b\'Did the U.S. Prep Georgia for War with Russia?\'

In [48]:
# clean the dataset 
clean_headlines = []

for i in range(0,len(headlines)):
    # replace non-alphabetic characters with spaces
    clean_headlines.append(re.sub("b[(')]", ' ', headlines[i])) # remove b'
    clean_headlines[i] = re.sub('b[(")]', ' ', clean_headlines[i]) # remove b"
    clean_headlines[i] = re.sub("[\ ']", ' ', clean_headlines[i]) # remove \'

<>:8: SyntaxWarning: invalid escape sequence '\ '
<>:8: SyntaxWarning: invalid escape sequence '\ '
C:\Users\35387\AppData\Local\Temp\ipykernel_23740\4110210003.py:8: SyntaxWarning: invalid escape sequence '\ '
  clean_headlines[i] = re.sub("[\ ']", ' ', clean_headlines[i]) # remove \'


In [49]:
# show the combined cleaned headlines 
clean_headlines[20]

' A French judge has ordered two branches of Scientologists and their leaders to stand trial for fraud     Russia in legal bid to ban South Park    60 Minutes  Cut Ahmadinejad s Statement,  Solution Is Democracy  in Israel/Palestine"  U.S. drones kill 13 in missile attack in Pakistan   Screw You, TSA: No Conviction on Key Charges in Liquid-Bomb Trial in London   Scientology on trial for fraud in France!   An EU ban on ads with sexist overtones? Another quasi-fictional piece of translucent flimflam   Film Backs Afghans Claims of US Killings [of 90+ civilians]   Giant Buddha found at Afghan site.   After denying strenously the US reopens inquiry into Afghan attack that may have killed upto 90 civilians   Videos surface showing dead Afghan children after US raid, sparking a new investigation   "Consortium" of Media Execs to Canadian Green Party:  You can\\ t participate in debate because the other parties don\\ t want you there.   Everything going wrong in the world .. in one convenient g

In [50]:
# add clean headlines to the merge dataset 
merge['Combined News'] = clean_headlines

merge['Combined News'][0]

' Georgia  downs two Russian warplanes  as countries move to brink of war"  BREAKING: Musharraf to be impeached.   Russia Today: Columns of troops roll into South Ossetia; footage from fighting (YouTube)   Russian tanks are moving towards the capital of South Ossetia, which has reportedly been completely destroyed by Georgian artillery fire   Afghan children raped with  impunity,  U.N. official says - this is sick, a three year old was raped and they do nothing"  150 Russian tanks have entered South Ossetia whilst Georgia shoots down two Russian jets.   Breaking: Georgia invades South Ossetia, Russia warned it would intervene on SO s side"  The  enemy combatent  trials are nothing but a sham: Salim Haman has been sentenced to 5 1/2 years, but will be kept longer anyway just because they feel like it."  Georgian troops retreat from S. Osettain capital, presumably leaving several hundred people killed. [VIDEO]   Did the U.S. Prep Georgia for War with Russia?   Rice Gives Green Light for 

In [51]:
merge.head(3)

,Date,Label,Top1,Top2,Top3,Top4,Top5,Top6,Top7,Top8,...,Top23,Top24,Top25,Open,High,Low,Close,Volume,Adj Close,Combined News
0,2008-08-08,0,"b""Georgia 'downs two Russian warplanes' as cou...",b'BREAKING: Musharraf to be impeached.',b'Russia Today: Columns of troops roll into So...,b'Russian tanks are moving towards the capital...,"b""Afghan children raped with 'impunity,' U.N. ...",b'150 Russian tanks have entered South Ossetia...,"b""Breaking: Georgia invades South Ossetia, Rus...","b""The 'enemy combatent' trials are nothing but...",...,b'Indian shoe manufactory - And again in a se...,b'Visitors Suffering from Mental Illnesses Ban...,"b""No Help for Mexico's Kidnapping Surge""",11432.089844,11759.959961,11388.040039,11734.320312,212830000,11734.320312,Georgia downs two Russian warplanes as coun...
1,2008-08-11,1,b'Why wont America and Nato help us? If they w...,b'Bush puts foot down on Georgian conflict',"b""Jewish Georgian minister: Thanks to Israeli ...",b'Georgian army flees in disarray as Russians ...,"b""Olympic opening ceremony fireworks 'faked'""",b'What were the Mossad with fraudulent New Zea...,b'Russia angered by Israeli military sale to G...,b'An American citizen living in S.Ossetia blam...,...,b'Perhaps *the* question about the Georgia - R...,b'Russia is so much better at war',"b""So this is what it's come to: trading sex fo...",11729.669922,11867.110352,11675.530273,11782.349609,183190000,11782.349609,Why wont America and Nato help us? If they wo...
2,2008-08-12,0,b'Remember that adorable 9-year-old who sang a...,"b""Russia 'ends Georgia operation'""","b'""If we had no sexual harassment we would hav...","b""Al-Qa'eda is losing support in Iraq because ...",b'Ceasefire in Georgia: Putin Outmaneuvers the...,b'Why Microsoft and Intel tried to kill the XO...,b'Stratfor: The Russo-Georgian War and the Bal...,"b""I'm Trying to Get a Sense of This Whole Geor...",...,b'Christopher King argues that the US and NATO...,b'America: The New Mexico?',"b""BBC NEWS | Asia-Pacific | Extinction 'by man...",11781.700195,11782.349609,11601.519531,11642.469727,173590000,11642.469727,Remember that adorable 9-year-old who sang at...


In [52]:
# subjectivity is a measure of how subjective or objective a text is, ranging from 0 (very objective) to 1 (very subjective) - if it is based on facts or opinions
def getSubjectivity(text):
    return TextBlob(text).sentiment.subjectivity  # textblob function to get subjectivity score which ranges from 0 to 1   

#polarity is a measure of how positive or negative a text is, ranging from -1 (very negative) to 1 (very positive)
def getPolarity(text):
    return TextBlob(text).sentiment.polarity

In [53]:
# create two new columns to add to the merged datasets - gets text from combined news column
merge['Subjectivity'] = merge['Combined News'].apply(getSubjectivity) # apply to function to the rows in the 'Combined News' column
merge['Polarity'] = merge['Combined News'].apply(getPolarity)

In [54]:
merge.head(3)   

,Date,Label,Top1,Top2,Top3,Top4,Top5,Top6,Top7,Top8,...,Top25,Open,High,Low,Close,Volume,Adj Close,Combined News,Subjectivity,Polarity
0,2008-08-08,0,"b""Georgia 'downs two Russian warplanes' as cou...",b'BREAKING: Musharraf to be impeached.',b'Russia Today: Columns of troops roll into So...,b'Russian tanks are moving towards the capital...,"b""Afghan children raped with 'impunity,' U.N. ...",b'150 Russian tanks have entered South Ossetia...,"b""Breaking: Georgia invades South Ossetia, Rus...","b""The 'enemy combatent' trials are nothing but...",...,"b""No Help for Mexico's Kidnapping Surge""",11432.089844,11759.959961,11388.040039,11734.320312,212830000,11734.320312,Georgia downs two Russian warplanes as coun...,0.267549,-0.048568
1,2008-08-11,1,b'Why wont America and Nato help us? If they w...,b'Bush puts foot down on Georgian conflict',"b""Jewish Georgian minister: Thanks to Israeli ...",b'Georgian army flees in disarray as Russians ...,"b""Olympic opening ceremony fireworks 'faked'""",b'What were the Mossad with fraudulent New Zea...,b'Russia angered by Israeli military sale to G...,b'An American citizen living in S.Ossetia blam...,...,"b""So this is what it's come to: trading sex fo...",11729.669922,11867.110352,11675.530273,11782.349609,183190000,11782.349609,Why wont America and Nato help us? If they wo...,0.374806,0.121956
2,2008-08-12,0,b'Remember that adorable 9-year-old who sang a...,"b""Russia 'ends Georgia operation'""","b'""If we had no sexual harassment we would hav...","b""Al-Qa'eda is losing support in Iraq because ...",b'Ceasefire in Georgia: Putin Outmaneuvers the...,b'Why Microsoft and Intel tried to kill the XO...,b'Stratfor: The Russo-Georgian War and the Bal...,"b""I'm Trying to Get a Sense of This Whole Geor...",...,"b""BBC NEWS | Asia-Pacific | Extinction 'by man...",11781.700195,11782.349609,11601.519531,11642.469727,173590000,11642.469727,Remember that adorable 9-year-old who sang at...,0.518785,-0.046530


In [55]:
# get the sentiment scores using SentimentIntensityAnalyser
def getSIA(text):
    sia = SentimentIntensityAnalyzer()
    sentiment = sia.polarity_scores(text)
    return sentiment

In [56]:
# get sentiment scroes for each day 
compound = [] # the score that calculates the combined scroe of the lexicon ratings 
negative = []
positive = []
neutral = []
SIA = 0

for i in range(0,len(merge['Combined News'])):
    SIA = getSIA(merge['Combined News'][i]) # takes in the merged text 
    compound.append(SIA['compound'])
    negative.append(SIA['neg'])
    neutral.append(SIA['neu'])
    positive.append(SIA['pos'])


In [57]:
# store the sentiment scores in the merged dataset 
merge['Compound'] = compound
merge['Negative'] = negative
merge['Neutral'] = neutral
merge['Positive'] = positive

In [58]:
# merge.head(3)

In [59]:
# collapse data to train our model on model 
# a list of columns to keep 
keep_columns = ['Open', 'High', 'Low', 'Volume', 'Subjectivity', 'Polarity', 'Compound', 'Negative', 'Neutral', 'Positive', 'Label']
# Define explicit feature columns (same order used for training and prediction)
features_columns = ['Open', 'High', 'Low', 'Volume', 'Subjectivity', 'Polarity', 'Compound', 'Negative', 'Neutral', 'Positive']
df = merge[keep_columns]
df.head(3)

,Open,High,Low,Volume,Subjectivity,Polarity,Compound,Negative,Neutral,Positive,Label
0,11432.089844,11759.959961,11388.040039,212830000,0.267549,-0.048568,-0.9982,0.233,0.726,0.041,0
1,11729.669922,11867.110352,11675.530273,183190000,0.374806,0.121956,-0.9858,0.188,0.724,0.088,1
2,11781.700195,11782.349609,11601.519531,173590000,0.518785,-0.046530,-0.9715,0.126,0.819,0.055,0


In [60]:
# create the feature dataset 
# Use explicit feature columns to ensure consistent ordering between training and prediction
X = dfX = df[features_columns].to_numpy()  # features as numpy array

# the target dataset - contained in the label column 
y = df['Label'].to_numpy()

In [61]:
# split the data into 80% training the model and 20% of the data will be for testing the model 
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [62]:
# create and train the model 
model = LinearDiscriminantAnalysis().fit(x_train, y_train)

In [71]:
# show the model predictions and a simple aggregated confidence metric
predictions = model.predict(x_test)

# Try to get class probabilities if the model supports it, otherwise fall back to proportion of predicted labels
import numpy as _np
try:
    probs = model.predict_proba(x_test) 
    # average probability across the test set for each class
    avg_prob = probs.mean(axis=0) 
    pred_label = int(_np.argmax(avg_prob))
    confidence = float(avg_prob[pred_label])
except Exception:
    # fallback: use fraction of predictions equal to the modal predicted label
    pred_label = int(_np.round(predictions.mean()))  # 1 if majority are 1 else 0
    confidence = float((predictions == pred_label).mean())

label_str = 'UP' if pred_label == 1 else 'DOWN'
pct = int(round(confidence * 100))
from datetime import datetime, timezone
ts = datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M GMT')
model_version = 'v1'
print(f'"Prediction: {label_str} ({pct}% probability) — Model {model_version}, generated {ts}"')

# also show raw predictions and a quick distribution for inspection
print('\nPrediction distribution (counts):')
unique, counts = _np.unique(predictions, return_counts=True)
print(dict(zip(unique.astype(int).tolist(), counts.tolist())))
predictions

"Prediction: UP (57% probability) — Model v1, generated 2025-11-24 11:17 GMT"

Prediction distribution (counts):
{0: 154, 1: 244}


array([1, 1, 0, 0, 0, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1,
       1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 0,
       0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0,
       1, 1, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1, 1,
       1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1,
       1, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1,
       1, 0, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1,
       1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1,
       1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1,
       0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1,
       1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 0,
       0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 0, 1,
       1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 0, 0, 1,
       0, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0,

In [ ]:
# add per row predictions to the dataset df using the trained model
# this cell appends Prediction (0/1) and Prediction probability
features = df[features_columns]
full_X = features.to_numpy()

try:
    full_probs = model.predict_proba(full_X)
    full_pred = full_probs.argmax(axis=1)
    full_conf = full_probs.max(axis=1)
except Exception:
    # if model doesn't support predict_proba it falls back to direct predict
    full_pred = model.predict(full_X)
    full_conf = None

# attach predictions to df
# keep numeric 0/1 for later use
df['Prediction'] = full_pred
if full_conf is not None:
    df['Prediction_Prob'] = full_conf

print('Added Prediction column to df; sample rows:')
from IPython.display import display
display(df.head(15))

Added Prediction column to df; sample rows:


C:\Users\35387\AppData\Local\Temp\ipykernel_23740\1360171697.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Prediction'] = full_pred
C:\Users\35387\AppData\Local\Temp\ipykernel_23740\1360171697.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Prediction_Prob'] = full_conf


,Open,High,Low,Volume,Subjectivity,Polarity,Compound,Negative,Neutral,Positive,Label,Prediction,Prediction_Prob
0,11432.089844,11759.959961,11388.040039,212830000,0.267549,-0.048568,-0.9982,0.233,0.726,0.041,0,1,0.994064
1,11729.669922,11867.110352,11675.530273,183190000,0.374806,0.121956,-0.9858,0.188,0.724,0.088,1,1,0.817760
2,11781.700195,11782.349609,11601.519531,173590000,0.518785,-0.046530,-0.9715,0.126,0.819,0.055,0,0,0.895032
3,11632.809570,11633.780273,11453.339844,182550000,0.364021,0.011398,-0.9809,0.143,0.793,0.064,0,0,0.913361
4,11532.070312,11718.280273,11450.889648,159790000,0.375099,0.040677,-0.9882,0.188,0.719,0.093,1,1,0.905058
5,11611.209961,11709.889648,11599.730469,215040000,0.457692,0.047756,-0.9880,0.173,0.753,0.074,1,1,0.815428
6,11659.650391,11690.429688,11434.120117,156290000,0.485995,0.016759,-0.9938,0.220,0.725,0.055,0,0,0.936402
7,11478.089844,11478.169922,11318.500000,171580000,0.345230,-0.025814,-0.9974,0.245,0.719,0.037,0,0,0.907154
8,11345.940430,11454.150391,11290.580078,144880000,0.218470,0.038384,-0.9913,0.180,0.760,0.060,1,1,0.754742
9,11415.230469,11476.209961,11315.570312,130020000,0.258333,0.049074,-0.9966,0.214,0.723,0.064,1,0,0.557735


In [65]:
y_test

array([1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 1, 1,
       1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1,
       0, 1, 1, 1, 0, 0, 0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0, 0, 1, 0, 0,
       0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 0, 1,
       1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1,
       1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 1, 0, 1, 1,
       1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 1, 1, 1,
       1, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1,
       0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1,
       0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1,
       1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0,
       0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 0, 1,
       1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1,
       0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0,

In [66]:
# show the model metrics
print(classification_report(y_test, predictions))

              precision    recall  f1-score   support

           0       0.85      0.77      0.81       171
           1       0.84      0.90      0.87       227

    accuracy                           0.84       398
   macro avg       0.84      0.83      0.84       398
weighted avg       0.84      0.84      0.84       398


